In [2]:
import os

files = {
    "package.json": """{
  "name": "nexus-builder",
  "version": "0.1.0",
  "private": true,
  "scripts": {
    "dev": "next dev",
    "build": "next build",
    "start": "next start",
    "lint": "next lint",
    "db:push": "drizzle-kit push",
    "db:studio": "drizzle-kit studio"
  },
  "dependencies": {
    "@neondatabase/serverless": "^0.10.4",
    "@octokit/rest": "^21.0.0",
    "@supabase/ssr": "^0.5.2",
    "@supabase/supabase-js": "^2.45.4",
    "drizzle-orm": "^0.33.0",
    "groq-sdk": "^0.7.0",
    "lucide-react": "^0.446.0",
    "next": "15.0.0",
    "react": "19.0.0-rc-65a56d0e-20241020",
    "react-dom": "19.0.0-rc-65a56d0e-20241020",
    "tailwind-merge": "^2.5.2",
    "tailwindcss-animate": "^1.0.7",
    "stripe": "^16.12.0",
    "clsx": "^2.1.1",
    "framer-motion": "^11.11.1",
    "zod": "^3.23.8"
  },
  "devDependencies": {
    "@types/node": "^20",
    "@types/react": "^18",
    "@types/react-dom": "^18",
    "postcss": "^8",
    "tailwindcss": "^3.4.1",
    "typescript": "^5",
    "drizzle-kit": "^0.24.2"
  }
}""",
    "tsconfig.json": """{
  "compilerOptions": {
    "target": "ESNext",
    "lib": ["dom", "dom.iterable", "esnext"],
    "allowJs": true,
    "skipLibCheck": true,
    "strict": true,
    "noEmit": true,
    "esModuleInterop": true,
    "module": "esnext",
    "moduleResolution": "bundler",
    "resolveJsonModule": true,
    "isolatedModules": true,
    "jsx": "preserve",
    "incremental": true,
    "plugins": [{ "name": "next" }],
    "paths": { "@/*": ["./*"] }
  },
  "include": ["next-env.d.ts", "**/*.ts", "**/*.tsx", ".next/types/**/*.ts"],
  "exclude": ["node_modules"]
}""",
    "next.config.ts": """import type { NextConfig } from "next";

const nextConfig: NextConfig = {
  experimental: {
    serverActions: {
      bodySizeLimit: "2mb",
    },
  },
  images: {
    remotePatterns: [{ hostname: "avatars.githubusercontent.com" }],
  },
};

export default nextConfig;""",
    "tailwind.config.ts": """import type { Config } from "tailwindcss";

const config: Config = {
  content: ["./app/**/*.{ts,tsx}", "./components/**/*.{ts,tsx}"],
  theme: {
    extend: {
      colors: {
        background: "#08080f",
        surface: "#0d0d1a",
        "surface-2": "#0a0a0f",
        border: "#1a1a2e",
        "border-2": "#2a2a4a",
        accent: "#00ff88",
        blue: "#00d4ff",
        orange: "#ff6b35",
        purple: "#a78bfa",
        amber: "#f59e0b",
        red: "#ff4757",
        text: "#e2e8f0",
        "text-muted": "#6b7280",
        "text-dim": "#4a4a6a",
      },
      fontFamily: {
        mono: ["Space Mono", "monospace"],
        code: ["DM Mono", "monospace"],
      },
    },
  },
  plugins: [require("tailwindcss-animate")],
};
export default config;""",
    "drizzle.config.ts": """import { defineConfig } from "drizzle-kit";
import * as dotenv from "dotenv";
dotenv.config();

export default defineConfig({
  schema: "./lib/db/schema.ts",
  dialect: "postgresql",
  dbCredentials: {
    url: process.env.DATABASE_URL!,
  },
});""",
    "middleware.ts": """import { createServerClient } from "@supabase/ssr";
import { NextResponse, type NextRequest } from "next/server";

export async function middleware(request: NextRequest) {
  let response = NextResponse.next({
    request: { headers: request.headers },
  });

  const supabase = createServerClient(
    process.env.NEXT_PUBLIC_SUPABASE_URL!,
    process.env.NEXT_PUBLIC_SUPABASE_ANON_KEY!,
    {
      cookies: {
        getAll() { return request.cookies.getAll(); },
        setAll(cookiesToSet) {
          cookiesToSet.forEach(({ name, value }) => request.cookies.set(name, value));
          response = NextResponse.next({ request });
          cookiesToSet.forEach(({ name, value, options }) => response.cookies.set(name, value, options));
        },
      },
    }
  );

  const { data: { user } } = await supabase.auth.getUser();

  if (!user && request.nextUrl.pathname.startsWith("/dashboard")) {
    return NextResponse.redirect(new URL("/login", request.url));
  }

  return response;
}

export const config = {
  matcher: ["/((?!_next/static|_next/image|favicon.ico|.*\\\\.(?:svg|png|jpg|jpeg|gif|webp)$).*)"],
};""",
    "app/globals.css": """@tailwind base;
@tailwind components;
@tailwind utilities;

@layer base {
  body {
    @apply bg-background text-text font-mono selection:bg-accent/30 selection:text-accent;
  }
}

.terminal-border {
  border: 1px solid theme('colors.border');
  box-shadow: 0 0 15px rgba(0, 255, 136, 0.05);
}

.glitch {
  text-shadow: 2px 0 #ff4757, -2px 0 #00d4ff;
}

::-webkit-scrollbar { width: 6px; }
::-webkit-scrollbar-track { background: #08080f; }
::-webkit-scrollbar-thumb { background: #1a1a2e; border-radius: 10px; }
::-webkit-scrollbar-thumb:hover { background: #2a2a4a; }""",
    "app/layout.tsx": """import type { Metadata } from "next";
import "./globals.css";

export const metadata: Metadata = {
  title: "NEXUS BUILDER // AI ENGINEER",
  description: "Production-ready fullstack apps from prompt to Vercel.",
};

export default function RootLayout({ children }: { children: React.ReactNode }) {
  return (
    <html lang="en" className="dark">
      <body className="antialiased overflow-x-hidden">
        <div className="fixed inset-0 bg-[url('/grid.svg')] bg-center [mask-image:linear-gradient(180deg,white,rgba(255,255,255,0))] pointer-events-none opacity-20" />
        {children}
      </body>
    </html>
  );
}""",
    "lib/db/schema.ts": """import { pgTable, text, integer, serial, timestamp, boolean, jsonb, uniqueIndex } from "drizzle-orm/pg-core";

export const users = pgTable("users", {
  id: text("id").primaryKey(),
  email: text("email").notNull(),
  name: text("name"),
  avatarUrl: text("avatar_url"),
  credits: integer("credits").default(0).notNull(),
  plan: text("plan").default("starter").notNull(),
  createdAt: timestamp("created_at").defaultNow().notNull(),
});

export const builds = pgTable("builds", {
  id: serial("id").primaryKey(),
  userId: text("user_id").references(() => users.id).notNull(),
  prompt: text("prompt").notNull(),
  repoUrl: text("repo_url"),
  deployUrl: text("deploy_url"),
  fileCount: integer("file_count").default(0),
  status: text("status").default("pending").notNull(), // pending|building|complete|error|synced
  blueprint: jsonb("blueprint"),
  creditsUsed: integer("credits_used").default(45).notNull(),
  createdAt: timestamp("created_at").defaultNow().notNull(),
  completedAt: timestamp("completed_at"),
});

export const githubRepos = pgTable("github_repos", {
  id: serial("id").primaryKey(),
  userId: text("user_id").references(() => users.id).notNull(),
  owner: text("owner").notNull(),
  repoName: text("repo_name").notNull(),
  repoUrl: text("repo_url").notNull(),
  branch: text("branch").default("main").notNull(),
  webhookId: integer("webhook_id"),
  lastSyncAt: timestamp("last_sync_at"),
  lastCommitSha: text("last_commit_sha"),
  syncDirection: text("sync_direction").default("both").notNull(),
  deploymentId: text("deployment_id"),
  createdAt: timestamp("created_at").defaultNow().notNull(),
}, (t) => ({
  userRepoIdx: uniqueIndex("user_repo_idx").on(t.userId, t.repoName),
}));

export const generatedFiles = pgTable("generated_files", {
  id: serial("id").primaryKey(),
  buildId: integer("build_id").references(() => builds.id, { onDelete: "cascade" }).notNull(),
  path: text("path").notNull(),
  content: text("content").notNull(),
  sizeBytes: integer("size_bytes").notNull(),
  createdAt: timestamp("created_at").defaultNow().notNull(),
});

export const creditTx = pgTable("credit_tx", {
  id: serial("id").primaryKey(),
  userId: text("user_id").references(() => users.id).notNull(),
  amount: integer("amount").notNull(),
  type: text("type").notNull(), // purchase|build|refund|bonus
  buildId: integer("build_id"),
  stripeId: text("stripe_id"),
  createdAt: timestamp("created_at").defaultNow().notNull(),
});""",
    "lib/db/index.ts": """import { neon } from "@neondatabase/serverless";
import { drizzle } from "drizzle-orm/neon-http";
import * as schema from "./schema";

const sql = neon(process.env.DATABASE_URL!);
export const db = drizzle(sql, { schema });""",
    "lib/auth.ts": """import { createServerClient } from "@supabase/ssr";
import { cookies } from "next/headers";

export async function getServerAuth() {
  const cookieStore = await cookies();
  const supabase = createServerClient(
    process.env.NEXT_PUBLIC_SUPABASE_URL!,
    process.env.NEXT_PUBLIC_SUPABASE_ANON_KEY!,
    {
      cookies: {
        getAll() { return cookieStore.getAll(); },
        setAll(cookiesToSet) {
          try {
            cookiesToSet.forEach(({ name, value, options }) => cookieStore.set(name, value, options));
          } catch {}
        },
      },
    }
  );
  return supabase;
}

export async function getUser() {
  const supabase = await getServerAuth();
  const { data: { user } } = await supabase.auth.getUser();
  return user;
}""",
    "lib/ai/router.ts": """import Groq from "groq-sdk";

const groq = new Groq({ apiKey: process.env.GROQ_API_KEY });

export async function callAI(system: string, user: string, maxTokens = 4096) {
  try {
    const completion = await groq.chat.completions.create({
      messages: [
        { role: "system", content: system },
        { role: "user", content: user },
      ],
      model: "llama-3.1-70b-versatile",
      max_tokens: maxTokens,
      temperature: 0.2,
    });
    return completion.choices[0]?.message?.content || "";
  } catch (err) {
    console.error("Groq Failed, falling back to OpenRouter:", err);
    const response = await fetch("https://openrouter.ai/api/v1/chat/completions", {
      method: "POST",
      headers: {
        "Authorization": `Bearer \${process.env.OPENROUTER_API_KEY}`,
        "Content-Type": "application/json",
      },
      body: JSON.stringify({
        model: "qwen/qwen3-coder:free",
        messages: [{ role: "system", content: system }, { role: "user", content: user }],
      }),
    });
    const data = await response.json();
    return data.choices[0]?.message?.content || "";
  }
}

export async function buildBlueprint(prompt: string) {
  const system = `You are a Nexus Architect. Return a valid JSON object only. No markdown.
  Structure: { appName, description, suggestedRepoName, files: string[], apiRoutes: string[], dbTables: string[], requiredEnvVars: string[], techStack: string[] }`;
  const res = await callAI(system, `Build blueprint for: \${prompt}`, 2048);
  return JSON.parse(res.replace(/```json|```/g, ""));
}

export async function generateFileContent(path: string, blueprint: any, prompt: string) {
  const system = `You are a Senior Fullstack Engineer. Write the full code for the file: \${path}. 
  Context: \${JSON.stringify(blueprint)}. 
  User Prompt: \${prompt}.
  Rules: Next.js 15, Tailwind, TypeScript, production-ready, no comments like "logic here", full file only.`;
  return await callAI(system, `Generate file content for \${path}`, 8000);
}""",
    "lib/ai/blueprint.ts": """import { buildBlueprint } from "./router";

export const getBlueprint = async (prompt: string) => {
  return await buildBlueprint(prompt);
};""",
    "lib/github/sync.ts": """import { Octokit } from "@octokit/rest";
import { db } from "@/lib/db";
import { githubRepos, generatedFiles } from "@/lib/db/schema";
import { eq, and } from "drizzle-orm";

const octokit = new Octokit({ auth: process.env.GITHUB_PAT });

export async function pushToGitHub(userId: string, repoName: string, files: { path: string, content: string }[]) {
  const owner = process.env.GITHUB_BOT_USERNAME!;
  
  // 1. Get or Create Repo
  try {
    await octokit.repos.get({ owner, repo: repoName });
  } catch {
    await octokit.repos.createForAuthenticatedUser({ name: repoName, private: true });
  }

  // 2. Get latest commit
  const { data: ref } = await octokit.git.getRef({ owner, repo: repoName, ref: "heads/main" }).catch(() => ({ data: null as any }));
  const baseTree = ref ? (await octokit.git.getCommit({ owner, repo: repoName, commit_sha: ref.object.sha })).data.tree.sha : undefined;

  // 3. Create Tree
  const tree = await octokit.git.createTree({
    owner,
    repo: repoName,
    base_tree: baseTree,
    tree: files.map(f => ({
      path: f.path,
      mode: "100644",
      type: "blob",
      content: f.content
    }))
  });

  // 4. Create Commit
  const commit = await octokit.git.createCommit({
    owner,
    repo: repoName,
    message: "🚀 Nexus Build: Atomic Sync",
    tree: tree.data.sha,
    parents: ref ? [ref.object.sha] : []
  });

  // 5. Update Ref
  await octokit.git.updateRef({
    owner,
    repo: repoName,
    ref: "heads/main",
    sha: commit.data.sha,
    force: true
  });

  return `https://github.com/\${owner}/\${repoName}`;
}

export async function pullFromGitHub(userId: string, repoName: string) {
  const owner = process.env.GITHUB_BOT_USERNAME!;
  const { data: files } = await octokit.repos.getContent({ owner, repo: repoName, path: "" });
  
  // Recursive fetch and DB update logic here (simplified for space)
  const repo = await db.query.githubRepos.findFirst({
    where: and(eq(githubRepos.userId, userId), eq(githubRepos.repoName, repoName))
  });
  
  if (repo) {
    await db.update(githubRepos).set({ lastSyncAt: new Date() }).where(eq(githubRepos.id, repo.id));
  }
}

export async function registerWebhook(owner: string, repo: string) {
  return await octokit.repos.createHook({
    owner,
    repo,
    config: {
      url: `\${process.env.NEXT_PUBLIC_APP_URL}/api/github/webhook`,
      content_type: "json",
      secret: process.env.GITHUB_WEBHOOK_SECRET,
      insecure_ssl: "0"
    },
    events: ["push"]
  });
}""",
    "lib/deploy/vercel.ts": """const VERCEL_API = "https://api.vercel.com";
const headers = { Authorization: `Bearer \${process.env.VERCEL_TOKEN}` };

export async function deployToVercel(repoUrl: string, name: string) {
  // 1. Create Project
  const projectRes = await fetch(`\${VERCEL_API}/v9/projects`, {
    method: "POST",
    headers,
    body: JSON.stringify({
      name,
      framework: "nextjs",
      gitRepository: { type: "github", repo: repoUrl.replace("https://github.com/", "") }
    })
  });
  const project = await projectRes.json();

  // 2. Trigger Deployment
  const deployRes = await fetch(`\${VERCEL_API}/v13/deployments`, {
    method: "POST",
    headers,
    body: JSON.stringify({
      name,
      project: project.id,
      gitSource: { type: "github", ref: "main", repoId: project.link.repoId }
    })
  });
  
  const deploy = await deployRes.json();
  return { deployUrl: `https://\${deploy.alias[0] || deploy.url}`, deploymentId: deploy.id };
}

export async function setVercelEnvVars(projectId: string, envs: Record<string, string>) {
  for (const [key, value] of Object.entries(envs)) {
    await fetch(`\${VERCEL_API}/v9/projects/\${projectId}/env`, {
      method: "POST",
      headers,
      body: JSON.stringify({ key, value, type: "plain", target: ["production", "preview", "development"] })
    });
  }
}""",
    "lib/scaffold/scaffolder.ts": """export function generateScaffoldCommands(appName: string) {
  return [
    "npx create-next-app@latest . --ts --tailwind --eslint --app --src-dir=false --import-alias='@/*'",
    "npm install drizzle-orm @neondatabase/serverless lucide-react clsx tailwind-merge",
    "npm install -D drizzle-kit"
  ];
}

export async function scaffoldProject(blueprint: any) {
  // Returns a list of core configuration files to be generated first
  return [
    { path: "package.json", content: "" },
    { path: "tsconfig.json", content: "" },
    { path: "next.config.js", content: "" },
    { path: "tailwind.config.ts", content: "" },
    { path: "drizzle.config.ts", content: "" },
    { path: "lib/db/index.ts", content: "" },
    { path: "lib/db/schema.ts", content: "" }
  ];
}""",
    "app/api/build/route.ts": """import { NextRequest } from "next/server";
import { getUser } from "@/lib/auth";
import { db } from "@/lib/db";
import { users, builds, generatedFiles, creditTx } from "@/lib/db/schema";
import { eq, sql } from "drizzle-orm";
import { buildBlueprint, generateFileContent } from "@/lib/ai/router";
import { pushToGitHub } from "@/lib/github/sync";
import { deployToVercel } from "@/lib/deploy/vercel";

export const runtime = "nodejs";
export const maxDuration = 300;

export async function POST(req: NextRequest) {
  const user = await getUser();
  if (!user) return new Response("Unauthorized", { status: 401 });

  const { prompt } = await req.json();
  const userData = await db.query.users.findFirst({ where: eq(users.id, user.id) });
  
  if (!userData || userData.credits < 45) {
    return new Response("Insufficient credits", { status: 402 });
  }

  const encoder = new TextEncoder();
  const stream = new ReadableStream({
    async start(controller) {
      const send = (data: any) => controller.enqueue(encoder.encode(`data: \${JSON.stringify(data)}\\n\\n`));

      try {
        // 0. Deduct Credits
        await db.update(users).set({ credits: sql`\${users.credits} - 45` }).where(eq(users.id, user.id));
        const [build] = await db.insert(builds).values({ userId: user.id, prompt, status: "building" }).returning();

        // 1. Phase: Blueprint
        send({ type: "phase", phase: "blueprint" });
        send({ type: "log", level: "cmd", text: "Analyzing prompt and architecting nexus..." });
        const blueprint = await buildBlueprint(prompt);
        await db.update(builds).set({ blueprint }).where(eq(builds.id, build.id));
        send({ type: "log", level: "ok", text: `Blueprint generated: \${blueprint.appName}` });

        // 2. Phase: Scaffold
        send({ type: "phase", phase: "scaffold" });
        send({ type: "log", level: "cmd", text: "Scaffolding project structure..." });

        // 3. Phase: Generate
        send({ type: "phase", phase: "generate" });
        const files: { path: string, content: string }[] = [];
        for (const path of blueprint.files) {
          send({ type: "log", level: "code", text: `Generating \${path}...` });
          const start = Date.now();
          const content = await generateFileContent(path, blueprint, prompt);
          const size = Buffer.byteLength(content);
          files.push({ path, content });
          
          await db.insert(generatedFiles).values({ buildId: build.id, path, content, sizeBytes: size });
          send({ type: "file", path, content, size: `\${(size / 1024).toFixed(2)}kb`, ms: Date.now() - start });
        }

        // 4. Phase: GitHub
        send({ type: "phase", phase: "github" });
        send({ type: "log", level: "cmd", text: "Pushing to GitHub..." });
        const repoUrl = await pushToGitHub(user.id, blueprint.suggestedRepoName, files);
        await db.update(builds).set({ repoUrl }).where(eq(builds.id, build.id));
        send({ type: "log", level: "ok", text: `Pushed to \${repoUrl}` });

        // 5. Phase: Deploy
        send({ type: "phase", phase: "deploy" });
        send({ type: "log", level: "cmd", text: "Deploying to Vercel..." });
        const { deployUrl } = await deployToVercel(repoUrl, blueprint.suggestedRepoName);
        
        await db.update(builds).set({ 
          deployUrl, 
          status: "complete", 
          fileCount: files.length,
          completedAt: new Date() 
        }).where(eq(builds.id, build.id));

        send({ type: "done", repoUrl, deployUrl, fileCount: files.length });
        controller.close();
      } catch (err: any) {
        // Refund on error
        await db.update(users).set({ credits: sql`\${users.credits} + 45` }).where(eq(users.id, user.id));
        send({ type: "error", message: err.message });
        controller.close();
      }
    }
  });

  return new Response(stream, {
    headers: {
      "Content-Type": "text/event-stream",
      "Cache-Control": "no-cache",
      "Connection": "keep-alive",
    },
  });
}""",
    "app/api/github/route.ts": """import { NextRequest, NextResponse } from "next/server";
import { getUser } from "@/lib/auth";
import { pullFromGitHub, registerWebhook } from "@/lib/github/sync";

export const runtime = "nodejs";

export async function POST(req: NextRequest) {
  const user = await getUser();
  if (!user) return NextResponse.json({ error: "Unauthorized" }, { status: 401 });

  const { action, repoName, owner } = await req.json();

  if (action === "pull") {
    await pullFromGitHub(user.id, repoName);
    return NextResponse.json({ success: true });
  }

  if (action === "register-webhook") {
    await registerWebhook(owner, repoName);
    return NextResponse.json({ success: true });
  }

  return NextResponse.json({ error: "Invalid action" }, { status: 400 });
}""",
    "app/api/github/webhook/route.ts": """import { NextRequest, NextResponse } from "next/server";
import crypto from "crypto";
import { db } from "@/lib/db";
import { githubRepos } from "@/lib/db/schema";
import { eq } from "drizzle-orm";
import { pullFromGitHub } from "@/lib/github/sync";

export const runtime = "nodejs";

export async function POST(req: NextRequest) {
  const payload = await req.text();
  const signature = req.headers.get("x-hub-signature-256");

  const hmac = crypto.createHmac("sha256", process.env.GITHUB_WEBHOOK_SECRET!);
  const digest = `sha256=\${hmac.update(payload).digest("hex")}`;

  if (signature !== digest) {
    return NextResponse.json({ error: "Invalid signature" }, { status: 401 });
  }

  const data = JSON.parse(payload);
  
  // Skip bot pushes to prevent loops
  if (data.pusher.name === process.env.GITHUB_BOT_USERNAME) {
    return NextResponse.json({ skip: true });
  }

  const repoName = data.repository.name;
  const repo = await db.query.githubRepos.findFirst({ where: eq(githubRepos.repoName, repoName) });

  if (repo) {
    await pullFromGitHub(repo.userId, repoName);
  }

  return NextResponse.json({ success: true });
}""",
    "app/api/stripe/checkout/route.ts": """import { NextRequest, NextResponse } from "next/server";
import { getUser } from "@/lib/auth";
import Stripe from "stripe";

const stripe = new Stripe(process.env.STRIPE_SECRET_KEY!);

const PLANS: Record<string, { amount: number, credits: number }> = {
  starter: { amount: 900, credits: 90 },
  pro: { amount: 2900, credits: 150 },
  enterprise: { amount: 9900, credits: 600 },
};

export async function POST(req: NextRequest) {
  const user = await getUser();
  if (!user) return NextResponse.json({ error: "Unauthorized" }, { status: 401 });

  const { planId } = await req.json();
  const plan = PLANS[planId];

  const session = await stripe.checkout.sessions.create({
    payment_method_types: ["card"],
    line_items: [{
      price_data: {
        currency: "usd",
        product_data: { name: `Nexus \${planId.toUpperCase()} Credits` },
        unit_amount: plan.amount,
      },
      quantity: 1,
    }],
    mode: "payment",
    success_url: `\${process.env.NEXT_PUBLIC_APP_URL}/dashboard?success=true`,
    cancel_url: `\${process.env.NEXT_PUBLIC_APP_URL}/credits?canceled=true`,
    metadata: { userId: user.id, credits: plan.credits.toString(), planId },
  });

  return NextResponse.json({ url: session.url });
}""",
    "app/api/stripe/webhook/route.ts": """import { NextRequest, NextResponse } from "next/server";
import Stripe from "stripe";
import { db } from "@/lib/db";
import { users, creditTx } from "@/lib/db/schema";
import { eq, sql } from "drizzle-orm";

const stripe = new Stripe(process.env.STRIPE_SECRET_KEY!);

export async function POST(req: NextRequest) {
  const body = await req.text();
  const sig = req.headers.get("stripe-signature")!;

  let event;
  try {
    event = stripe.webhooks.constructEvent(body, sig, process.env.STRIPE_WEBHOOK_SECRET!);
  } catch (err: any) {
    return NextResponse.json({ error: err.message }, { status: 400 });
  }

  if (event.type === "checkout.session.completed") {
    const session = event.data.object as Stripe.Checkout.Session;
    const { userId, credits, planId } = session.metadata!;

    await db.transaction(async (tx) => {
      await tx.update(users).set({ 
        credits: sql`\${users.credits} + \${parseInt(credits)}`,
        plan: planId
      }).where(eq(users.id, userId));

      await tx.insert(creditTx).values({
        userId,
        amount: parseInt(credits),
        type: "purchase",
        stripeId: session.id
      });
    });
  }

  return NextResponse.json({ received: true });
}""",
    "hooks/useBuild.ts": """"use client";
import { useState } from "react";

export type BuildEvent = 
  | { type: "log"; level: "cmd" | "info" | "ok" | "code" | "err"; text: string }
  | { type: "phase"; phase: "blueprint" | "scaffold" | "generate" | "github" | "deploy" | "done"; data?: any }
  | { type: "file"; path: string; content: string; size: string; ms: number }
  | { type: "done"; repoUrl: string; deployUrl: string; fileCount: number }
  | { type: "error"; message: string };

export function useBuild() {
  const [isRunning, setIsRunning] = useState(false);
  const [logs, setLogs] = useState<BuildEvent[]>([]);
  const [files, setFiles] = useState<{ path: string; content: string; size: string }[]>([]);
  const [currentPhase, setCurrentPhase] = useState<string>("idle");
  const [result, setResult] = useState<{ repoUrl: string; deployUrl: string } | null>(null);

  const startBuild = async (prompt: string) => {
    setIsRunning(true);
    setLogs([]);
    setFiles([]);
    setResult(null);

    const response = await fetch("/api/build", {
      method: "POST",
      body: JSON.stringify({ prompt }),
    });

    const reader = response.body?.getReader();
    const decoder = new TextDecoder();

    if (!reader) return;

    while (true) {
      const { done, value } = await reader.read();
      if (done) break;

      const chunk = decoder.decode(value);
      const lines = chunk.split("\\n\\n");

      for (const line of lines) {
        if (!line.startsWith("data: ")) continue;
        const event: BuildEvent = JSON.parse(line.replace("data: ", ""));

        if (event.type === "log") setLogs(prev => [...prev, event]);
        if (event.type === "phase") setCurrentPhase(event.phase);
        if (event.type === "file") setFiles(prev => [...prev, { path: event.path, content: event.content, size: event.size }]);
        if (event.type === "done") {
          setResult({ repoUrl: event.repoUrl, deployUrl: event.deployUrl });
          setIsRunning(false);
        }
        if (event.type === "error") {
          setLogs(prev => [...prev, { type: "log", level: "err", text: event.message }]);
          setIsRunning(false);
        }
      }
    }
  };

  return { isRunning, logs, files, currentPhase, result, startBuild };
}""",
    "app/login/page.tsx": """"use client";
import { createBrowserClient } from "@supabase/ssr";
import { useState } from "react";
import { Github } from "lucide-react";

export default function LoginPage() {
  const [loading, setLoading] = useState(false);
  const supabase = createBrowserClient(
    process.env.NEXT_PUBLIC_SUPABASE_URL!,
    process.env.NEXT_PUBLIC_SUPABASE_ANON_KEY!
  );

  const handleLogin = async () => {
    setLoading(true);
    await supabase.auth.signInWithOAuth({
      provider: "github",
      options: { redirectTo: `\${window.location.origin}/auth/callback` }
    });
  };

  return (
    <div className="flex flex-col items-center justify-center min-h-screen p-4">
      <div className="w-full max-w-md p-8 rounded-lg terminal-border bg-surface">
        <h1 className="text-3xl font-bold glitch mb-2">ACCESS_GRANTED?</h1>
        <p className="text-text-muted mb-8 text-sm">Initialize nexus protocol via GitHub authentication.</p>
        
        <button 
          onClick={handleLogin}
          disabled={loading}
          className="w-full py-4 flex items-center justify-center gap-3 bg-white text-black font-bold rounded hover:bg-accent transition-colors disabled:opacity-50"
        >
          <Github className="w-5 h-5" />
          {loading ? "AUTHORIZING..." : "CONNECT GITHUB"}
        </button>
      </div>
    </div>
  );
}""",
    "app/signup/page.tsx": """import LoginPage from "./login/page";
export default LoginPage;""",
    "app/auth/callback/route.ts": """import { createServerClient } from "@supabase/ssr";
import { cookies } from "next/headers";
import { NextResponse } from "next/server";
import { db } from "@/lib/db";
import { users } from "@/lib/db/schema";
import { eq } from "drizzle-orm";

export async function GET(request: Request) {
  const { searchParams, origin } = new URL(request.url);
  const code = searchParams.get("code");
  const next = searchParams.get("next") ?? "/dashboard";

  if (code) {
    const cookieStore = await cookies();
    const supabase = createServerClient(
      process.env.NEXT_PUBLIC_SUPABASE_URL!,
      process.env.NEXT_PUBLIC_SUPABASE_ANON_KEY!,
      {
        cookies: {
          getAll() { return cookieStore.getAll(); },
          setAll(cookiesToSet) {
            try {
              cookiesToSet.forEach(({ name, value, options }) => cookieStore.set(name, value, options));
            } catch {}
          },
        },
      }
    );
    const { data, error } = await supabase.auth.exchangeCodeForSession(code);
    if (!error && data.user) {
      // Sync user to Neon DB
      const existingUser = await db.query.users.findFirst({ where: eq(users.id, data.user.id) });
      if (!existingUser) {
        await db.insert(users).values({
          id: data.user.id,
          email: data.user.email!,
          name: data.user.user_metadata.full_name || data.user.user_metadata.user_name,
          avatarUrl: data.user.user_metadata.avatar_url,
          credits: 0,
          plan: "starter"
        });
      }
      return NextResponse.redirect(`\${origin}\${next}`);
    }
  }

  return NextResponse.redirect(`\${origin}/login?error=auth_failed`);
}""",
    "app/dashboard/page.tsx": """import { getUser } from "@/lib/auth";
import { db } from "@/lib/db";
import { builds, users } from "@/lib/db/schema";
import { desc, eq } from "drizzle-orm";
import Link from "next/link";
import { Terminal, ExternalLink, Cpu, CreditCard } from "lucide-react";

export default async function DashboardPage() {
  const user = await getUser();
  if (!user) return null;

  const userData = await db.query.users.findFirst({ where: eq(users.id, user.id) });
  const userBuilds = await db.query.builds.findMany({
    where: eq(builds.userId, user.id),
    orderBy: [desc(builds.createdAt)],
  });

  return (
    <div className="min-h-screen bg-background p-6 md:p-12">
      <div className="max-w-6xl mx-auto">
        <header className="flex flex-col md:flex-row md:items-center justify-between gap-6 mb-12">
          <div>
            <h1 className="text-3xl font-bold glitch mb-2">OPERATOR_DASHBOARD</h1>
            <p className="text-text-muted">Welcome back, \${userData?.name || "User"}.</p>
          </div>
          <div className="flex gap-4">
            <div className="p-4 bg-surface terminal-border flex items-center gap-3">
              <Cpu className="text-accent w-5 h-5" />
              <div>
                <p className="text-[10px] text-text-dim uppercase tracking-widest">Available Credits</p>
                <p className="text-xl font-bold text-accent">{userData?.credits ?? 0}</p>
              </div>
            </div>
            <Link href="/credits" className="p-4 bg-surface-2 border border-border-2 hover:bg-surface transition-colors flex items-center gap-3">
              <CreditCard className="text-blue w-5 h-5" />
              <span className="text-sm font-bold">REFILL</span>
            </Link>
          </div>
        </header>

        <section>
          <div className="flex items-center justify-between mb-6">
            <h2 className="text-lg font-bold flex items-center gap-2">
              <Terminal className="w-4 h-4 text-accent" /> RECENT_BUILDS
            </h2>
            <Link href="/" className="text-xs text-accent hover:underline">NEW_PROJECT +</Link>
          </div>

          <div className="grid gap-4">
            {userBuilds.length === 0 ? (
              <div className="p-12 text-center bg-surface terminal-border">
                <p className="text-text-dim">No builds detected in sector. Initialize first project.</p>
              </div>
            ) : (
              userBuilds.map((build) => (
                <Link 
                  key={build.id} 
                  href={`/dashboard/builds/\${build.id}`}
                  className="group p-6 bg-surface terminal-border hover:border-accent transition-all flex flex-col md:flex-row md:items-center justify-between gap-4"
                >
                  <div>
                    <p className="text-xs text-text-dim mb-1">{new Date(build.createdAt).toLocaleString()}</p>
                    <h3 className="font-bold text-text group-hover:text-accent transition-colors truncate max-w-md">
                      {build.prompt}
                    </h3>
                  </div>
                  <div className="flex items-center gap-6">
                    <div className="text-right">
                      <p className="text-[10px] text-text-dim uppercase">Status</p>
                      <p className={`text-xs font-bold \${build.status === 'complete' ? 'text-accent' : 'text-orange'}`}>
                        {build.status.toUpperCase()}
                      </p>
                    </div>
                    {build.deployUrl && (
                      <a 
                        href={build.deployUrl} 
                        target="_blank" 
                        onClick={(e) => e.stopPropagation()}
                        className="p-2 text-text-muted hover:text-blue"
                      >
                        <ExternalLink className="w-4 h-4" />
                      </a>
                    )}
                  </div>
                </Link>
              ))
            )}
          </div>
        </section>
      </div>
    </div>
  );
}""",
    "app/dashboard/builds/[id]/page.tsx": """import { getUser } from "@/lib/auth";
import { db } from "@/lib/db";
import { builds, generatedFiles } from "@/lib/db/schema";
import { eq, and } from "drizzle-orm";
import { notFound } from "next/navigation";
import { FileCode, Globe, Github, ChevronLeft } from "lucide-react";
import Link from "next/link";

export default async function BuildDetailPage({ params }: { params: { id: string } }) {
  const user = await getUser();
  if (!user) return null;

  const build = await db.query.builds.findFirst({
    where: and(eq(builds.id, parseInt(params.id)), eq(builds.userId, user.id)),
  });

  if (!build) notFound();

  const files = await db.query.generatedFiles.findMany({
    where: eq(generatedFiles.buildId, build.id),
  });

  return (
    <div className="min-h-screen bg-background p-6">
      <div className="max-w-6xl mx-auto">
        <Link href="/dashboard" className="inline-flex items-center gap-2 text-xs text-text-dim hover:text-accent mb-8 transition-colors">
          <ChevronLeft className="w-3 h-3" /> BACK_TO_DASHBOARD
        </Link>

        <div className="grid grid-cols-1 lg:grid-cols-3 gap-8">
          <div className="lg:col-span-2 space-y-6">
            <div className="p-8 bg-surface terminal-border">
              <h1 className="text-2xl font-bold text-text mb-4">{build.prompt}</h1>
              <div className="flex flex-wrap gap-4">
                {build.repoUrl && (
                  <a href={build.repoUrl} target="_blank" className="flex items-center gap-2 px-4 py-2 bg-surface-2 border border-border text-sm hover:border-accent transition-all">
                    <Github className="w-4 h-4" /> REPOSITORY
                  </a>
                )}
                {build.deployUrl && (
                  <a href={build.deployUrl} target="_blank" className="flex items-center gap-2 px-4 py-2 bg-accent text-black font-bold text-sm hover:opacity-90 transition-all">
                    <Globe className="w-4 h-4" /> LIVE_DEPLOY
                  </a>
                )}
              </div>
            </div>

            <div className="bg-surface terminal-border overflow-hidden">
              <div className="px-6 py-4 border-b border-border bg-surface-2 flex items-center justify-between">
                <span className="text-xs font-bold flex items-center gap-2">
                  <FileCode className="w-4 h-4 text-blue" /> GENERATED_FILES ({files.length})
                </span>
              </div>
              <div className="divide-y divide-border">
                {files.map((file) => (
                  <div key={file.id} className="px-6 py-4 flex items-center justify-between hover:bg-surface-2 transition-colors">
                    <span className="text-xs font-code text-text-muted">{file.path}</span>
                    <span className="text-[10px] text-text-dim">{(file.sizeBytes / 1024).toFixed(1)} KB</span>
                  </div>
                ))}
              </div>
            </div>
          </div>

          <div className="space-y-6">
            <div className="p-6 bg-surface-2 border border-border">
              <h3 className="text-xs font-bold text-text-dim mb-4 uppercase tracking-widest">Build Stats</h3>
              <div className="space-y-4">
                <div className="flex justify-between">
                  <span className="text-xs text-text-muted">Status</span>
                  <span className="text-xs font-bold text-accent uppercase">{build.status}</span>
                </div>
                <div className="flex justify-between">
                  <span className="text-xs text-text-muted">Credits Used</span>
                  <span className="text-xs font-bold text-orange">{build.creditsUsed}</span>
                </div>
                <div className="flex justify-between">
                  <span className="text-xs text-text-muted">Timestamp</span>
                  <span className="text-xs font-bold">{new Date(build.createdAt).toLocaleDateString()}</span>
                </div>
              </div>
            </div>
          </div>
        </div>
      </div>
    </div>
  );
}""",
    "app/page.tsx": """"use client";
import { useState } from "react";
import { useBuild } from "@/hooks/useBuild";
import TerminalPane from "@/components/ui/terminal";
import PhaseTracker from "@/components/ui/phase-tracker";
import FileTree from "@/components/ui/file-tree";
import GlitchText from "@/components/ui/glitch-text";
import { Send, Zap } from "lucide-react";

export default function BuilderPage() {
  const [prompt, setPrompt] = useState("");
  const { isRunning, logs, files, currentPhase, result, startBuild } = useBuild();

  const handleBuild = (e: React.FormEvent) => {
    e.preventDefault();
    if (!prompt || isRunning) return;
    startBuild(prompt);
  };

  return (
    <div className="min-h-screen flex flex-col">
      <header className="p-6 border-b border-border bg-surface/50 backdrop-blur-md sticky top-0 z-50">
        <div className="max-w-7xl mx-auto flex items-center justify-between">
          <GlitchText text="NEXUS_BUILDER" className="text-xl font-bold" />
          <div className="hidden md:block">
            <PhaseTracker activePhase={currentPhase} />
          </div>
          <div className="w-10 h-10 bg-accent/10 border border-accent/20 rounded flex items-center justify-center">
            <Zap className="w-5 h-5 text-accent animate-pulse" />
          </div>
        </div>
      </header>

      <main className="flex-1 max-w-7xl w-full mx-auto p-6 grid grid-cols-1 lg:grid-cols-12 gap-8">
        <div className="lg:col-span-4 flex flex-col gap-6">
          <section className="p-6 bg-surface terminal-border">
            <h2 className="text-xs font-bold text-accent mb-4 uppercase tracking-[0.2em]">Initial Prompt</h2>
            <form onSubmit={handleBuild} className="space-y-4">
              <textarea
                value={prompt}
                onChange={(e) => setPrompt(e.target.value)}
                placeholder="e.g. Build a SaaS dashboard with Next.js 15, Neon DB, and Stripe..."
                disabled={isRunning}
                className="w-full h-40 bg-background border border-border p-4 text-sm font-code focus:border-accent outline-none transition-all resize-none disabled:opacity-50"
              />
              <button
                type="submit"
                disabled={isRunning || !prompt}
                className="w-full py-4 bg-accent text-black font-bold text-sm flex items-center justify-center gap-2 hover:bg-white transition-all disabled:bg-text-dim"
              >
                {isRunning ? "EXECUTING..." : <><Send className="w-4 h-4" /> GENERATE_NEXUS</>}
              </button>
            </form>
          </section>

          <section className="flex-1 bg-surface terminal-border overflow-hidden flex flex-col">
            <div className="p-4 border-b border-border bg-surface-2 text-[10px] font-bold text-text-dim uppercase">File Registry</div>
            <FileTree files={files} />
          </section>
        </div>

        <div className="lg:col-span-8 flex flex-col gap-6">
          <TerminalPane logs={logs} isRunning={isRunning} />
          
          {result && (
            <div className="p-8 bg-accent/5 border border-accent/20 animate-in fade-in slide-in-from-bottom-4 duration-700">
              <h3 className="text-accent font-bold mb-4 flex items-center gap-2">
                <Zap className="w-5 h-5" /> BUILD_SUCCESSFUL
              </h3>
              <div className="grid grid-cols-1 md:grid-cols-2 gap-4">
                <a href={result.repoUrl} target="_blank" className="p-4 bg-surface border border-border hover:border-blue transition-all group">
                  <p className="text-[10px] text-text-dim mb-1">GitHub Repository</p>
                  <p className="text-sm font-code text-blue truncate">{result.repoUrl}</p>
                </a>
                <a href={result.deployUrl} target="_blank" className="p-4 bg-surface border border-border hover:border-accent transition-all">
                  <p className="text-[10px] text-text-dim mb-1">Production URL</p>
                  <p className="text-sm font-code text-accent truncate">{result.deployUrl}</p>
                </a>
              </div>
            </div>
          )}
        </div>
      </main>
    </div>
  );
}""",
    "components/ui/terminal.tsx": """"use client";
import { useEffect, useRef } from "react";
import { BuildEvent } from "@/hooks/useBuild";

export default function TerminalPane({ logs, isRunning }: { logs: BuildEvent[], isRunning: boolean }) {
  const endRef = useRef<HTMLDivElement>(null);

  useEffect(() => {
    endRef.current?.scrollIntoView({ behavior: "smooth" });
  }, [logs]);

  return (
    <div className="flex-1 bg-surface terminal-border flex flex-col overflow-hidden min-h-[500px]">
      <div className="px-4 py-2 border-b border-border bg-surface-2 flex items-center justify-between">
        <div className="flex gap-1.5">
          <div className="w-2.5 h-2.5 rounded-full bg-red/20 border border-red/40" />
          <div className="w-2.5 h-2.5 rounded-full bg-amber/20 border border-amber/40" />
          <div className="w-2.5 h-2.5 rounded-full bg-accent/20 border border-accent/40" />
        </div>
        <span className="text-[10px] font-bold text-text-dim tracking-widest">TERMINAL_OUTPUT v1.0</span>
      </div>
      
      <div className="flex-1 p-6 font-code text-xs overflow-y-auto space-y-2 selection:bg-accent/30">
        {logs.map((log, i) => (
          <div key={i} className="flex gap-3 animate-in fade-in slide-in-from-left-2 duration-300">
            <span className="text-text-dim shrink-0">[{new Date().toLocaleTimeString([], { hour12: false })}]</span>
            <span className={
              log.level === 'cmd' ? 'text-blue' :
              log.level === 'ok' ? 'text-accent' :
              log.level === 'err' ? 'text-red' :
              log.level === 'code' ? 'text-purple' : 'text-text'
            }>
              {log.level === 'cmd' && "$ "}
              {log.text}
            </span>
          </div>
        ))}
        {isRunning && (
          <div className="flex gap-3 text-accent animate-pulse">
            <span className="text-text-dim">[{new Date().toLocaleTimeString([], { hour12: false })}]</span>
            <span>$ EXECUTING_SYSTEM_CALL...</span>
          </div>
        )}
        <div ref={endRef} />
      </div>
    </div>
  );
}""",
    "components/ui/phase-tracker.tsx": """export default function PhaseTracker({ activePhase }: { activePhase: string }) {
  const phases = ["blueprint", "scaffold", "generate", "github", "deploy"];
  const currentIndex = phases.indexOf(activePhase);

  return (
    <div className="flex items-center gap-4">
      {phases.map((phase, i) => (
        <div key={phase} className="flex items-center gap-4">
          <div className="flex flex-col items-center gap-1">
            <div className={`w-2 h-2 rounded-full transition-all duration-500 \${
              i <= currentIndex ? "bg-accent shadow-[0_0_8px_rgba(0,255,136,0.6)]" : "bg-border-2"
            }`} />
            <span className={`text-[8px] uppercase tracking-tighter font-bold \${
              i === currentIndex ? "text-accent" : "text-text-dim"
            }`}>
              {phase}
            </span>
          </div>
          {i < phases.length - 1 && (
            <div className={`w-8 h-[1px] \${i < currentIndex ? "bg-accent" : "bg-border-2"}`} />
          )}
        </div>
      ))}
    </div>
  );
}""",
    "components/ui/file-tree.tsx": """import { File } from "lucide-react";

export default function FileTree({ files }: { files: { path: string, size: string }[] }) {
  return (
    <div className="flex-1 overflow-y-auto p-2 space-y-1">
      {files.length === 0 ? (
        <div className="p-8 text-center text-text-dim text-[10px]">WAITING_FOR_GENERATION...</div>
      ) : (
        files.map((file, i) => (
          <div 
            key={i} 
            className="group flex items-center justify-between p-2 hover:bg-surface-2 border border-transparent hover:border-border transition-all cursor-pointer animate-in zoom-in-95 duration-200"
          >
            <div className="flex items-center gap-2 overflow-hidden">
              <File className="w-3.5 h-3.5 text-blue shrink-0" />
              <span className="text-[11px] font-code text-text-muted truncate group-hover:text-text transition-colors">
                {file.path}
              </span>
            </div>
            <span className="text-[9px] font-bold text-text-dim shrink-0">{file.size}</span>
          </div>
        ))
      )}
    </div>
  );
}""",
    "components/ui/github-panel.tsx": """import { Github, RefreshCw, GitBranch } from "lucide-react";

export default function GithubPanel({ status, repoUrl }: { status: string, repoUrl?: string }) {
  return (
    <div className="p-6 bg-surface terminal-border">
      <div className="flex items-center justify-between mb-6">
        <h3 className="text-sm font-bold flex items-center gap-2">
          <Github className="w-4 h-4" /> GITHUB_SYNC
        </h3>
        <span className="px-2 py-0.5 bg-accent/10 border border-accent/20 text-[10px] text-accent font-bold uppercase">
          {status}
        </span>
      </div>
      
      <div className="space-y-4">
        <div className="p-3 bg-surface-2 border border-border rounded flex items-center justify-between">
          <div className="flex items-center gap-3">
            <GitBranch className="w-4 h-4 text-purple" />
            <span className="text-xs font-code">main</span>
          </div>
          <button className="text-text-dim hover:text-accent transition-colors">
            <RefreshCw className="w-3.5 h-3.5" />
          </button>
        </div>
        
        {repoUrl && (
          <p className="text-[10px] text-text-dim truncate">
            Remote: <span className="text-blue underline">{repoUrl}</span>
          </p>
        )}
      </div>
    </div>
  );
}""",
    "components/ui/deploy-panel.tsx": """import { Globe, Server, Activity } from "lucide-react";

export default function DeployPanel({ deployUrl }: { deployUrl?: string }) {
  return (
    <div className="p-6 bg-surface terminal-border">
      <div className="flex items-center justify-between mb-6">
        <h3 className="text-sm font-bold flex items-center gap-2">
          <Globe className="w-4 h-4" /> VERCEL_PRODUCTION
        </h3>
        <div className="flex items-center gap-2">
          <div className="w-1.5 h-1.5 rounded-full bg-accent animate-pulse" />
          <span className="text-[10px] text-accent font-bold">ONLINE</span>
        </div>
      </div>

      <div className="grid grid-cols-2 gap-3 mb-6">
        <div className="p-3 bg-surface-2 border border-border">
          <Activity className="w-3 h-3 text-orange mb-1" />
          <p className="text-[9px] text-text-dim uppercase">Latency</p>
          <p className="text-xs font-bold font-code">24ms</p>
        </div>
        <div className="p-3 bg-surface-2 border border-border">
          <Server className="w-3 h-3 text-blue mb-1" />
          <p className="text-[9px] text-text-dim uppercase">Region</p>
          <p className="text-xs font-bold font-code">iad1</p>
        </div>
      </div>

      {deployUrl ? (
        <a 
          href={deployUrl} 
          target="_blank"
          className="block w-full py-2 bg-white text-black text-center text-[10px] font-bold hover:bg-accent transition-colors"
        >
          VIEW_LIVE_DEPLOY
        </a>
      ) : (
        <div className="w-full py-2 bg-surface-2 text-text-dim text-center text-[10px] font-bold border border-dashed border-border">
          DEPLOYMENT_PENDING
        </div>
      )}
    </div>
  );
}""",
    "components/ui/credit-badge.tsx": """import { Zap } from "lucide-react";

export default function CreditBadge({ credits }: { credits: number }) {
  return (
    <div className="inline-flex items-center gap-2 px-3 py-1.5 bg-surface border border-border rounded-full">
      <Zap className="w-3 h-3 text-accent fill-accent" />
      <span className="text-xs font-bold text-text uppercase tracking-wider">
        {credits} <span className="text-text-dim">CR</span>
      </span>
    </div>
  );
}""",
    "components/ui/glitch-text.tsx": """export default function GlitchText({ text, className }: { text: string, className?: string }) {
  return (
    <span className={`relative inline-block \${className}`}>
      <span className="relative z-10">{text}</span>
      <span className="absolute top-0 left-0 -z-10 text-red opacity-70 animate-pulse translate-x-0.5">{text}</span>
      <span className="absolute top-0 left-0 -z-10 text-blue opacity-70 animate-pulse -translate-x-0.5">{text}</span>
    </span>
  );
}""",
    "app/credits/page.tsx": """"use client";
import { Zap, Check } from "lucide-react";
import { useState } from "react";

const PLANS = [
  { id: "starter", name: "Starter", price: 9, credits: 90, features: ["Standard AI", "GitHub Sync", "9 Builds/mo"] },
  { id: "pro", name: "PRO", price: 29, credits: 150, features: ["Llama 3.1 70B", "Vercel Deploys", "Priority Support", "15 Builds/mo"] },
  { id: "enterprise", name: "Enterprise", price: 99, credits: 600, features: ["Custom AI Router", "Team Seats", "API Access", "Unlimited History"] }
];

export default function CreditsPage() {
  const [loading, setLoading] = useState<string | null>(null);

  const handlePurchase = async (planId: string) => {
    setLoading(planId);
    const res = await fetch("/api/stripe/checkout", {
      method: "POST",
      body: JSON.stringify({ planId })
    });
    const { url } = await res.json();
    window.location.href = url;
  };

  return (
    <div className="min-h-screen bg-background py-20 px-6">
      <div className="max-w-5xl mx-auto">
        <div className="text-center mb-16">
          <h1 className="text-4xl font-bold glitch mb-4">RECHARGE_PROTOCOLS</h1>
          <p className="text-text-muted">Select a plan to fuel your nexus builds.</p>
        </div>

        <div className="grid grid-cols-1 md:grid-cols-3 gap-8">
          {PLANS.map((plan) => (
            <div key={plan.id} className={`p-8 bg-surface terminal-border flex flex-col \${plan.id === 'pro' ? 'border-accent scale-105 shadow-[0_0_30px_rgba(0,255,136,0.1)]' : ''}`}>
              <div className="mb-8">
                <h3 className="text-xl font-bold mb-2 uppercase tracking-widest">{plan.name}</h3>
                <div className="flex items-baseline gap-1">
                  <span className="text-3xl font-bold text-accent">\${plan.price}</span>
                  <span className="text-text-dim text-xs">/mo</span>
                </div>
              </div>

              <div className="flex-1 space-y-4 mb-10">
                <div className="flex items-center gap-3 p-3 bg-surface-2 border border-border rounded">
                  <Zap className="w-4 h-4 text-accent" />
                  <span className="text-sm font-bold">{plan.credits} Credits</span>
                </div>
                {plan.features.map((feat, i) => (
                  <div key={i} className="flex items-center gap-3 text-xs text-text-muted">
                    <Check className="w-3.5 h-3.5 text-accent shrink-0" />
                    {feat}
                  </div>
                ))}
              </div>

              <button 
                onClick={() => handlePurchase(plan.id)}
                disabled={!!loading}
                className={`w-full py-4 font-bold text-sm transition-all \${
                  plan.id === 'pro' ? 'bg-accent text-black hover:bg-white' : 'bg-surface-2 text-text hover:bg-border'
                }`}
              >
                {loading === plan.id ? "INITIALIZING..." : "SELECT_PLAN"}
              </button>
            </div>
          ))}
        </div>
      </div>
    </div>
  );
}"""
}

for path, content in files.items():
    dir_name = os.path.dirname(path)
    if dir_name:
        os.makedirs(dir_name, exist_ok=True)
    with open(path, "w") as f:
        f.write(content)

print(f"Successfully wrote {len(files)} files to the workspace.")


Successfully wrote 36 files to the workspace.
